Merge PFAS properties data

In [7]:
import pandas as pd

path_file = 'C:/Users/dell/Desktop/pfas/'

path_epa = path_file + "epa/epa.csv"
path_norman = path_file + "norman/norman.csv"
path_paper = path_file + "paper/paper.csv"

path_po_raw = path_file + "pfas_raw.csv"


df_epa = pd.read_csv(path_epa)
df_epa = df_epa[['CAS', 'boiling_point', 'melting_point', 'density', 'log_Koa', 
                 'log_Kow', 'solubility', 'pKa', 'logD5_5', 'logD7_4']]
df_epa = df_epa[df_epa['CAS'].notna()]
df_norman = pd.read_csv(path_norman)
df_norman = df_norman[['CAS', 'po_xlogp', 'log_Koc', 'log_Kow']]
df_norman = df_norman[df_norman['CAS'].notna()]
df_paper = pd.read_csv(path_paper)
df_paper = df_paper[['CAS', 'log_Kow', 'log_Kaw', 'log_Koa_wet', 'log_Koil_w', 'log_Koil_air','log_KHxd_air']]
df_paper = df_paper[df_paper['CAS'].notna()]
df_pubchem = pd.read_csv(path_file + "pubchem/pubchem.csv")
df_pubchem = df_pubchem[['CAS', 'po_m_w', 'po_xlogp']]
df_pubchem = df_pubchem[df_pubchem['CAS'].notna()]
df_po_raw = pd.read_csv(path_po_raw)

In [ ]:
def merge_chemical_properties(df_po_raw, df_paper, df_epa, df_pubchem, df_norman):
    result = df_po_raw.copy()

    suffixes = {
        'paper': '_paper',
        'epa': '_epa',
        'pubchem': '_pubchem',
        'norman': '_norman'
    }
    
    result = result.merge(df_paper, on='CAS', how='left', suffixes=('', suffixes['paper']))

    result = result.merge(df_epa, on='CAS', how='left', suffixes=('', suffixes['epa']))

    result = result.merge(df_pubchem, on='CAS', how='left', suffixes=('', suffixes['pubchem']))

    result = result.merge(df_norman, on='CAS', how='left', suffixes=('', suffixes['norman']))

    duplicate_properties = {
        'log_Kow': ['paper', 'epa', 'norman'],
        'log_Koa': ['paper', 'epa'],
        'po_xlogp': ['pubchem', 'norman']
    }

    for prop, sources in duplicate_properties.items():
        if prop not in result.columns:
            result[prop] = None

        for source in sources:
            source_col = f"{prop}{suffixes[source]}"
            if source_col in result.columns:
                
                result[prop] = result[prop].fillna(result[source_col])
                
                result = result.drop(columns=[source_col])
    
    return result


merged_df = merge_chemical_properties(df_po_raw, df_paper, df_epa, df_pubchem, df_norman)
merged_df = merged_df.rename(columns={'po_xlogp': 'logPx','pKa':'log_pKa'})
merged_df.to_csv(path_file + "pfas.csv", index=False)

### epa comptox

In [ ]:
import pandas as pd
import os
import glob

def process_csv_files(directory_path):
    csv_files = glob.glob(os.path.join(directory_path, "*.csv"))

    all_dfs = []
    
    for file in csv_files:

        filename = os.path.basename(file)
        dtxsid = filename.split('-')[0]   
        df = pd.read_csv(file)
        df.columns = df.columns.str.lower().str.replace(' ', '_')
        df['DTXSID'] = dtxsid
        all_dfs.append(df)

    final_df = pd.concat(all_dfs, ignore_index=True)

    cols = ['DTXSID'] + [col for col in final_df.columns if col != 'DTXSID']
    final_df = final_df[cols]
    
    return final_df

directory_path = r"C:\Users\dell\Desktop\pfas\epa\raw"
result_df = process_csv_files(directory_path)

result_df.to_csv( r"C:\Users\dell\Desktop\pfas\epa\merged_data.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np


chemical_list = pd.read_csv(r"C:\Users\dell\Desktop\pfas\epa\chemical_list.csv")
merged_data = pd.read_csv(r"C:\Users\dell\Desktop\pfas\epa\merged_data.csv")


merged_data = merged_data.merge(
    chemical_list[['DTXSID', 'CASRN']], 
    left_on='DTXSID', 
    right_on='DTXSID', 
    how='left'
)


merged_data = merged_data.rename(columns={'CASRN': 'CAS'})


if 'DTXSID' in merged_data.columns:
    merged_data = merged_data.drop('DTXSID', axis=1)


merged_data['experimental_median'] = merged_data['experimental_median'].replace('-', np.nan)


merged_data['final_data'] = np.where(
    merged_data['experimental_median'].notna(),
    merged_data['experimental_median'],
    merged_data['predicted_median']
)

properties = merged_data['property'].unique()

new_df = pd.DataFrame()
new_df['CAS'] = merged_data['CAS'].unique()


for prop in properties:

    prop_data = merged_data[merged_data['property'] == prop]

    prop_values = dict(zip(prop_data['CAS'], prop_data['final_data']))

    new_df[prop] = new_df['CAS'].map(prop_values)


new_df = new_df.rename(columns={'Boiling Point': 'boiling_point', 
                                'Melting Point': 'melting_point', 
                                'Density': 'density', 'LogKoa: Octanol-Air':'log_Koa',
                                'LogKow: Octanol-Water':'log_Kow', 'Water Solubility':'solubility',
                                'pKa Acidic Apparent':'pKa', 'LogD5.5':'logD5_5', 'LogD7.4':'logD7_4'})
new_df.to_csv(r"C:\Users\dell\Desktop\pfas\epa\epa.csv", index=False)

### norman

In [ ]:
import pandas as pd
import numpy as np


df = pd.read_csv(r'C:\Users\dell\Desktop\pfas\norman\raw\susdat_2025-02-20-084113.csv')


columns_mapping = {
    'CAS_RN_PubChem': 'CAS',
    'StdInChI': 'inchi',
    'PubChem_CID': 'CID',
    'DTXSID': 'DTXSID',
    'average_mass': 'po_m_w',
    'logKow_EPISuite': 'log_kow',
    'xlogp_ChemSpider': 'po_xlogp',
    'Koc_min_predicted (L/kg)': 'koc_min',
    'Koc_max_predicted (L/kg)': 'koc_max'
}


df = df[list(columns_mapping.keys())].rename(columns=columns_mapping)


def calculate_log_koc(row):
    if row['koc_min'] == row['koc_max']:
        return np.log10(row['koc_min'])
    else:
        return np.log10((row['koc_min'] + row['koc_max']) / 2)

df['log_koc'] = df.apply(calculate_log_koc, axis=1)


df.to_csv(r'C:\Users\dell\Desktop\pfas\norman\norman.csv', index=False)

### pubchem

In [ ]:
import pandas as pd
import pubchempy as pcp


file_path = r'C:\Users\dell\Desktop\pfas\pfas_raw.csv'
df = pd.read_csv(file_path)
df = df[df["CID"].notna()]
df[['CID']] = df[['CID']].astype(int)

cids = df['CID'].tolist()



result_df = pd.DataFrame(columns=['CID', 'CAS', 'MolecularWeight', 'MolecularFormula', 'XLogP',
                                  'InChI', 'ExactMass', 'IUPACName', 'Complexity', 'HeavyAtomCount',
                                  'IsomericSmiles'])


for cid in cids:

    compound = pcp.Compound.from_cid(cid)


    if compound:
        result_df = result_df.append({
            'CID': cid,
            'CAS': compound.cid,
            'MolecularWeight': compound.molecular_weight,
            'MolecularFormula': compound.molecular_formula,
            'XLogP': compound.xlogp,
            'InChI': compound.inchi,
            'ExactMass': compound.exact_mass,
            'IUPACName': compound.iupac_name,
            'Complexity': compound.complexity,
            'HeavyAtomCount': compound.heavy_atom_count,
            'IsomericSmiles': compound.isomeric_smiles
        }, ignore_index=True)
    else:

        result_df = result_df.append({
            'CID': cid,
            'CAS': None,
            'MolecularWeight': None,
            'MolecularFormula': None,
            'XLogP': None,
            'InChI': None,
            'ExactMass': None,
            'IUPACName': None,
            'Complexity': None,
            'HeavyAtomCount': None,
            'IsomericSmiles': None
        }, ignore_index=True)


result_file_path = r'C:\Users\dell\Desktop\pfas\pubchem\pubchem_raw.csv'
result_df.to_csv(result_file_path, index=False)


In [10]:
result_file_path = r'C:\Users\dell\Desktop\pfas\pubchem\pubchem_raw.csv'
df_pubchem_raw = pd.read_csv(result_file_path)

df_pubchem_raw = df_pubchem_raw[['CID', 'MolecularWeight', 'XLogP']]

file_path = r'C:\Users\dell\Desktop\pfas\pfas_raw.csv'
df_po = pd.read_csv(file_path)
df_po = df_po[df_po['CAS'].notna()]

po_dict = dict(zip(df_po['CID'], df_po['CAS']))

df_pubchem_raw['CAS'] = df_pubchem_raw['CID'].map(po_dict)
df_pubchem_raw = df_pubchem_raw.rename(columns={'MolecularWeight': 'po_m_w', 'XLogP': 'po_xlogp'})


df_pubchem_raw.to_csv(r'C:\Users\dell\Desktop\pfas\pubchem\pubchem.csv', index=False)